In [1]:
import sys
sys.path.append('../')

import torch
import json

from src.model import Transformer, generate_batch
from src.dataset import (
    NMRDataset, 
    load_multiplicity_codes, 
    load_split, 
    collate_fn
)

import sentencepiece as spm

from functools import partial

In [2]:
tokenizer = spm.SentencePieceProcessor('../checkpoints/nmr2struct.model')

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
print(device)

cuda


In [5]:
model = Transformer(
    multiplicity_vocab_size=len(NMRDataset.MULTIPLICITY2IDX),
    multiplicity_hidden_dim=48,
    spectrum_d_ff=256,
    spectrum_hidden_dim=104,
    intensity_d_ff=256,
    intensity_hidden_dim=104,
    tgt_vocab_size=tokenizer.vocab_size(),
    d_model=256,
    num_heads=8,
    num_layers=6,
    d_ff=512,
    max_seq_length=400,
    dropout=0.2,
    smiles_pad_token=tokenizer.pad_id(),
)
model.load_state_dict(torch.load('../checkpoints/bimodal_256_new_data_full_155.pt', map_location=device))
model = model.to(device)
model.eval()

Transformer(
  (spectrum_embedding): SpectrumEmbedding(
    (spectrum_embs): ModuleDict(
      (C_NMR): FourierEmbedding(
        (proj): Linear(in_features=256, out_features=104, bias=True)
      )
      (H_NMR): FourierEmbedding(
        (proj): Linear(in_features=256, out_features=104, bias=True)
      )
    )
    (intensity_embs): ModuleDict(
      (C_NMR): FourierEmbedding(
        (proj): Linear(in_features=256, out_features=104, bias=True)
      )
      (H_NMR): FourierEmbedding(
        (proj): Linear(in_features=256, out_features=104, bias=True)
      )
    )
    (multiplicity_embs): ModuleDict(
      (C_NMR): Embedding(10, 48)
      (H_NMR): Embedding(10, 48)
    )
  )
  (decoder_embedding): Embedding(512, 256)
  (positional_encoding): PositionalEncoding()
  (encoder_carbon_layers): ModuleList(
    (0-5): 6 x EncoderLayer(
      (self_attn): MultiHeadAttention(
        (W_q): Linear(in_features=256, out_features=256, bias=True)
        (W_k): Linear(in_features=256, out_featu

In [6]:
multiplicity_codes = load_multiplicity_codes('../data/multiplicity_codes.json')

In [7]:
test_data = load_split(
    path='../data/test_full_50k.jsonl', 
    multiplicity_codes=multiplicity_codes,
    solvent='CDCl3' # Так собраны данные
)

Loading test_full_50k.jsonl: 0it [00:00, ?it/s]

Processing spectra:   0%|          | 0/9954 [00:00<?, ?it/s]

In [10]:
# Для каждой молекулы есть разное количество 1H и 13C спектров
# Датасет устроен так что он каждый раз выдает случайную пару спектров
# И с вероятностью 20% маскирует один из спектров
# Если хочется работать иначе с этим - см. src/dataset.py L153

test_dataset = NMRDataset(
    data=test_data,
    solvent='CDCl3',
    tokenizer=tokenizer,
    smiles_bos_id=tokenizer.bos_id(),
    smiles_eos_id=tokenizer.eos_id(),
)

test_loader = torch.utils.data.DataLoader(
    test_dataset, 
    shuffle=False, 
    batch_size=16, 
    collate_fn=partial(
        collate_fn, 
        smiles_pad_id=tokenizer.pad_id(),
        spectrum_pad_token=-1000.,
        max_smiles_len=400
    )
)

In [16]:
from tqdm.auto import tqdm

all_predictions = []
all_true_smiles = []

for data in tqdm(test_loader):
    predictions = generate_batch(
        model=model,
        src_c_nmr=data["C_NMR"],
        src_h_nmr=data["H_NMR"],
        tokenizer=tokenizer,
        max_tokens=400,
        device=device,
    )

    true_smiles = tokenizer.decode(data["smiles"].squeeze().numpy().tolist())

    all_predictions.extend(predictions)
    all_true_smiles.extend(true_smiles)

  0%|          | 0/623 [00:00<?, ?it/s]

In [23]:
import json

assert len(all_predictions) == len(all_true_smiles)

results = [
    {"true_smiles": t, "predicted_smiles": p}
    for t, p in zip(all_true_smiles, all_predictions)
]

with open("test_predictions.json", "w") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

In [17]:
for pred_smiles, true_smiles in zip(all_predictions[:20], all_true_smiles[:20]):
    print(f"True SMILES: {true_smiles}", f"Predicted SMILES: {pred_smiles}", sep="\t")

True SMILES: O=C1OC(=O)c2ccccc21	Predicted SMILES: O=C1c2ccccc2C(=O)N1Cl
True SMILES: Clc1nc(Cl)c2ccccc2n1	Predicted SMILES: O=C(Cl)c1nc2ccccc2s1
True SMILES: c1ccc2c(N3CCCC3)ncnc2c1	Predicted SMILES: c1ccc2c(N3CCCC3)ncnc2c1
True SMILES: COc1ccc(C=O)cc1OC	Predicted SMILES: COc1ccc(C=O)cc1OC
True SMILES: COc1cc(C(O)c2cc(C3OCC(C)(C)CO3)cc(OC)c2OC)ccc1OCc1ccccc1	Predicted SMILES: COc1cc([C@H](O)c2cc(OC)c(OCc3ccccc3)cc2C2OCC(C)(C)CO2)ccc1OC
True SMILES: O=C(/C=C/C=C/c1ccc(F)cc1)Oc1cccc2cccc(OC(=O)/C=C/C=C/c3ccc(F)cc3)c12	Predicted SMILES: O=C1c2ccccc2C2(c3ccccc3)N(c3ccccc3)c3ccccc3C12c1ccccc1
True SMILES: CC(C)(C)OC(=O)N[C@H]1c2ccccc2C=C[C@H]1c1ccc(Cl)cc1	Predicted SMILES: CC(C)(C)OC(=O)N[C@H]1c2ccccc2C=C[C@@H]1c1ccc(Cl)cc1
True SMILES: CC(C)(C)OC(=O)N[C@H]1c2ccccc2C=C[C@H]1c1ccc(Br)cc1	Predicted SMILES: CC(C)(C)OC(=O)N[C@H]1c2ccccc2C=C[C@@H]1c1ccc(Br)cc1
True SMILES: CC(C)(C)OC(=O)N[C@H]1c2ccccc2C=C[C@H]1c1ccc([N+](=O)[O-])cc1	Predicted SMILES: CC(C)(C)OC(=O)N[C@H]1c2ccccc2C=C[C@H]1c1ccc(

In [27]:
from rdkit import Chem
from rdkit import RDLogger

RDLogger.DisableLog('rdApp.*')

def canonicalize_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol)

canon_true = [canonicalize_smiles(s) for s in all_true_smiles]
canon_pred = [canonicalize_smiles(s) for s in all_predictions]

valid_pred = sum(s is not None for s in canon_pred) / len(canon_pred)
canon_exact_match = sum(p == t for p, t in zip(canon_pred, canon_true)) / len(canon_true)

In [26]:
print("Validity:", valid_pred)
print("Canonical exact match:", canon_exact_match)

Validity: 0.9939722724532851
Canonical exact match: 0.39893510146674704
